# Goodness Of Fit, Independence, And Homogeneity

**Official MA1001B Alignment:** *6.8 goodness of fit; 6.9 independence; 6.10 homogeneity; 6.11 several proportions.*


## How To Use This Lesson

This notebook is designed as a guided teaching episode and interactive lab, not a passive code demonstration. To get the most out of this lesson:
1. **Read the conceptual explanations and explicit links** before running any code.
2. **Execute code cells sequentially**, paying attention to inline educational comments.
3. **Pause at the Guided Checkpoint** to discuss with a partner and write your reasoning before checking solutions.
4. **Complete the Independent Practice and Exit Ticket**; written justification is the primary evidence of statistical competence.


## Learning Goals

By the end of this lesson, you will be able to:
- Construct and inspect multi-category contingency tables using Pandas (`pd.crosstab`).
- Calculate row and column conditional percentage distributions to explore categorical associations.
- Execute Chi-Square tests of independence and homogeneity (`stats.chi2_contingency`) across multi-group data.
- Analyze residual cell differences (`observed - expected`) to identify specific category drivers of non-independence.


## The Three Explicit Links

In accordance with the MA1001B pedagogical framework, this lesson explicitly connects theory, computation, and action:

- **1. Conceptual Link (What is modeled):** We model categorical frequencies under null independence assumptions to determine whether consumer preferences require regional segmentation.
- **2. Computational Link (How Python represents it):** We use Pandas cross-tabulations and SciPy chi-square contingency evaluations to extract expected cell frequencies and residuals.
- **3. Decision Link (How it guides action):** Residual analysis highlights exactly which regional product preferences deviate from baseline, guiding targeted marketing campaigns.


## Decision Scenario

> **The Problem:** A regional manager wants to know whether customer preferences differ by region. If preferences are independent of region, a single strategy may be enough; otherwise segmentation may be needed.


## Conceptual Explanation

Categorical inference compares observed counts with expected counts under a null structure. Tests of independence ask whether two categorical variables are associated in one population. Tests of homogeneity compare distributions across groups.


## Mathematical Anchor

The chi-square statistic sums (observed - expected)^2 / expected across cells. Larger values indicate stronger disagreement with the null model.


## Data And Workflow Notes

Uses simulated region-choice data with a contingency table.


## Practical Python Workflow

The following worked example demonstrates how to implement these statistical concepts in Python to generate evidence for decision making.


### Step 1: Simulating Regional Preference Contingency Data

We simulate survey records for n=900 customers across three geographic regions (North, Center, South) choosing among three product packages (A, B, C).


In [ ]:
# Import required data science and statistical libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set reproducible random seed and visual styling
rng = np.random.default_rng(1001)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

# Simulate region and product choice records for n=900 customers
regions = rng.choice(["North", "Center", "South"], size=900, p=[0.35, 0.40, 0.25])
choice_probs = {
    "North": [0.52, 0.30, 0.18],  # North prefers Package A
    "Center": [0.42, 0.38, 0.20], # Center is balanced between A and B
    "South": [0.34, 0.42, 0.24],  # South prefers Package B
}
choices = [rng.choice(["Package_A", "Package_B", "Package_C"], p=choice_probs[r]) for r in regions]

survey = pd.DataFrame({"region": regions, "choice": choices})
table = pd.crosstab(survey["region"], survey["choice"], margins=True)
table


### Step 2: Row-Wise Conditional Preference Proportions

To see if product preferences differ by region, we normalize the contingency table by row totals to obtain conditional percentage distributions.


In [ ]:
# Calculate row-wise percentage distributions (excluding margins table for clean division)
raw_table = pd.crosstab(survey["region"], survey["choice"])
row_percentages = raw_table.div(raw_table.sum(axis=1), axis=0)
(row_percentages * 100).round(1).astype(str) + "%"


### Step 3: Chi-Square Test of Independence

We run the Chi-Square test of independence on the raw frequency counts to determine if the regional preference variations are statistically significant.


In [ ]:
# Execute Chi-Square test of independence
chi2_stat, p_value, dof, expected_matrix = stats.chi2_contingency(raw_table)

pd.Series({
    "chi_square_statistic": chi2_stat,
    "degrees_of_freedom_(r-1)*(c-1)": dof,
    "p_value": p_value,
    "reject_null_independence_(p<0.05)": p_value < 0.05
}).round(4)


### Step 4: Residual Analysis (Observed vs. Expected)

To understand *why* the test was significant, we construct a DataFrame of residual differences (`Observed - Expected`) to pinpoint which regional preferences drive the association.


In [ ]:
# Compare observed counts against expected null counts
expected_table = pd.DataFrame(expected_matrix, index=raw_table.index, columns=raw_table.columns)
residuals = raw_table - expected_table

print("--- Expected Counts under Null Independence ---")
display(expected_table.round(1))
print()
print("--- Residuals (Observed - Expected Counts) ---")
display(residuals.round(1))


## Guided Checkpoint

> [!IMPORTANT]
> **Pair Discussion & Writing Prompt:**
> Which specific cells in the residual table contribute most to the disagreement between observed data and expected null counts?

*Write your reasoned response below before continuing:*


## Common Mistakes & Statistical Pitfalls

Avoid these frequent errors when conducting or communicating this analysis:
- **Warning:** Conducting Chi-Square tests on percentages or proportions rather than raw integer frequency counts.
- **Warning:** Ignoring small expected cell counts (expected counts should generally be >= 5 for reliable chi-square approximations).
- **Warning:** Claiming that a significant chi-square test explains *why* categories are associated without inspecting cell residuals.


## Independent Practice

> [!TIP]
> **Your Task:**
> Write a regional marketing segmentation recommendation using both the row percentage table and the residual analysis results.

*Use the empty code and markdown cells below to implement your analysis and justify your recommendation.*


In [ ]:
# Write your independent practice code here
# Remember to inspect your outputs and check assumptions


## Decision Interpretation Template

Use this structured format to write your defensible conclusion and recommendation:

1. **The Decision Question:** *State the practical question being answered...*
2. **The Statistical Evidence:** *Summarize key metrics, intervals, p-values, or model comparisons...*
3. **Uncertainty & Limitations:** *Identify what the data cannot prove and what assumptions were made...*
4. **Actionable Recommendation:** *Therefore, I recommend [action] because [justification]...*


## Exit Ticket

> **Reflection:** What does an 'expected count' represent conceptually in a chi-square test of independence?

*Write your brief conceptual reflection below:*
